# Pipeline de Dados — Loja do League of Legends
## 01. Ingestão, Exploração, Limpeza e Normalização Relacional

**Disciplina:** Banco de Dados e Big Data para Data Science  
**Equipe:** Diogo Galrão Carvalho, Felipe Artur Macedo Lima, Luan Cavalcante Dias Rodrigues  

---

### Objetivo do Notebook
Este notebook realiza a extração dos dados brutos da API/catálogo de itens do *League of Legends* (`itens_raw.csv`), faz a limpeza e desaninhamento das estruturas semiestruturadas (como tags de categorias em JSON), normaliza o esquema para um **Modelo Relacional** consistente em 3ª Forma Normal (3FN) e gera entidades complementares para simulação de compras, partidas e perfis demográficos.

Esses dados serão utilizados em:
1. **UA 01 e UA 02:** Migração de banco relacional (PostgreSQL) para NoSQL orientado a documentos (MongoDB Atlas).
2. **UA 03 e UA 04:** Processamento analítico em larga escala com **Apache Spark** no Databricks e armazenamento em **Delta Lake**.

In [ ]:
import os
import json
import random
from datetime import datetime, timedelta
import pandas as pd

random.seed(42)
caminho_raw = '../data/raw/itens_raw.csv'

df_raw = pd.read_csv(caminho_raw)
print(f"Linhas e colunas carregadas: {df_raw.shape}")
df_raw.head(3)

### 1. Inspeção dos Dados Brutos
Verificamos os tipos de dados e a coluna semiestruturada `categorias_json`, que traz as tags de classificação de cada item em formato de lista JSON codificada como string.

In [ ]:
df_raw[['item_id', 'nome', 'preco_total_ouro', 'categorias_json']].info()
df_raw['categorias_json'].head()

### 2. Seleção de Atributos e Desaninhamento (Explode)
No modelo relacional, não podemos armazenar listas JSON diretamente dentro de uma coluna se quisermos manter a 1ª Forma Normal (1FN). Portanto, descompactamos (`explode`) cada tag de categoria para modelar o relacionamento Muitos-para-Muitos (N:N) entre `item` e `categoria`.

In [ ]:
colunas_selecionadas = ['item_id', 'nome', 'descricao_texto', 'categorias_json', 'preco_total_ouro']
df_itens = df_raw[colunas_selecionadas].copy()

# Converter strings JSON para listas Python reais e expandir
def parse_json_safely(val):
    try:
        res = json.loads(val)
        return res if res else ["Diversos"]
    except Exception:
        return ["Diversos"]

df_itens['categorias_json'] = df_itens['categorias_json'].apply(parse_json_safely)
df_itens_explodidos = df_itens[['item_id', 'categorias_json']].explode('categorias_json')
df_itens_explodidos.columns = ['item_id', 'nome_categoria']
df_itens_explodidos.head()

### 3. Normalização das Tabelas: `categoria` e `item_categoria` (N:N)
Criamos uma tabela de domínio de categorias únicas com chave primária substituta (`id_categoria`) e a tabela associativa `item_categoria` que implementa o N:N relacional.

In [ ]:
categorias_unicas = sorted(df_itens_explodidos['nome_categoria'].unique())
df_categoria = pd.DataFrame(categorias_unicas, columns=['nome_categoria']).reset_index()
df_categoria = df_categoria.rename(columns={'index': 'id_categoria'})
df_categoria['id_categoria'] += 1

# Mapear o ID da categoria de volta nos itens
df_item_categoria = pd.merge(
    df_itens_explodidos,
    df_categoria,
    on='nome_categoria',
    how='inner'
)[['item_id', 'id_categoria']].drop_duplicates().reset_index(drop=True)

print(f"Total de Categorias: {len(df_categoria)}")
print(f"Total de Relacionamentos Item-Categoria: {len(df_item_categoria)}")
df_categoria.head()

### 4. Tabela de Domínio dos Itens (`item`)
Limpamos o catálogo de itens, definindo o preço unitário em ouro (tratando itens gratuitos para valor simbólico de consumível) e a descrição.

In [ ]:
df_tabela_item = df_itens[['item_id', 'nome', 'preco_total_ouro', 'descricao_texto']].copy()
df_tabela_item = df_tabela_item.rename(columns={
    'preco_total_ouro': 'preco_unitario',
    'descricao_texto': 'descricao'
})

# Ajustar itens com preço 0 para 150 (valor padrão de consumível na simulação)
df_tabela_item['preco_unitario'] = df_tabela_item['preco_unitario'].apply(lambda x: 150 if x <= 0 else int(x))
df_tabela_item = df_tabela_item.drop_duplicates(subset=['item_id']).reset_index(drop=True)
print(f"Catálogo final de itens: {len(df_tabela_item)} itens cadastrados.")
df_tabela_item.head()

### 5. Tabela de Domínio de Elos (`elo`)
Elos competitivos do jogo ordenados do menor para o maior ranque.

In [ ]:
elos = [
    'Ferro', 'Bronze', 'Prata', 'Ouro', 'Platina',
    'Esmeralda', 'Diamante', 'Mestre', 'Grão-Mestre', 'Desafiante'
]
df_elo = pd.DataFrame(elos, columns=['elo']).reset_index()
df_elo = df_elo.rename(columns={'index': 'id_elo'})
df_elo['id_elo'] += 1
df_elo

### 6. Jogadores (`jogador`) e Fonte Complementar Demográfica para DBFS (`jogadores_demografia`)
Para a **UA 03 e UA 04**, a especificação exige uma **fonte complementar de dados** (ex: demografia de clientes/jogadores) a ser enviada ao DBFS para ser combinada no Spark via Broadcast Join. Criamos aqui tanto a tabela relacional do jogador quanto o dataset demográfico enriquecido.

In [ ]:
jogadores_raw = [
    (1, 'Faker', 10, 'KR', 'Coreia do Sul', 'Seul', 28, 'M', 'VIP', 'Desktop Gamer'),
    (2, 'ShowMaker', 9, 'KR', 'Coreia do Sul', 'Seul', 24, 'M', 'VIP', 'Desktop Gamer'),
    (3, 'Chovy', 10, 'KR', 'Coreia do Sul', 'Seul', 23, 'M', 'VIP', 'Desktop Gamer'),
    (4, 'Caps', 9, 'EUW', 'Dinamarca', 'Copenhague', 25, 'M', 'VIP', 'Desktop Gamer'),
    (5, 'Rekkles', 8, 'EUW', 'Suécia', 'Estocolmo', 27, 'M', 'VIP', 'Desktop Gamer'),
    (6, 'BrTT', 8, 'BR', 'Brasil', 'Rio de Janeiro', 33, 'M', 'VIP', 'Desktop Gamer'),
    (7, 'Kami', 7, 'BR', 'Brasil', 'Pelotas', 28, 'M', 'VIP', 'Desktop Gamer'),
    (8, 'Tinowns', 9, 'BR', 'Brasil', 'Campinas', 27, 'M', 'VIP', 'Desktop Gamer'),
    (9, 'Robo', 8, 'BR', 'Brasil', 'São Paulo', 26, 'M', 'VIP', 'Desktop Gamer'),
    (10, 'Cariok', 7, 'BR', 'Brasil', 'Rio de Janeiro', 25, 'M', 'Pro', 'Desktop Gamer'),
    (11, 'Titan', 8, 'BR', 'Brasil', 'Manaus', 24, 'M', 'VIP', 'Notebook Gamer'),
    (12, 'Aegis', 6, 'BR', 'Brasil', 'Curitiba', 23, 'M', 'Gratuito', 'Desktop Gamer'),
    (13, 'Ceos', 7, 'BR', 'Brasil', 'Belo Horizonte', 24, 'M', 'Gratuito', 'Desktop Gamer'),
    (14, 'Route', 8, 'KR', 'Coreia do Sul', 'Busan', 26, 'M', 'VIP', 'Desktop Gamer'),
    (15, 'Dynquedo', 6, 'BR', 'Brasil', 'Florianópolis', 27, 'M', 'Gratuito', 'Desktop Gamer'),
    (16, 'Jojo', 5, 'BR', 'Brasil', 'Porto Alegre', 25, 'M', 'Gratuito', 'Notebook Gamer'),
    (17, 'TitaN_Prime', 7, 'BR', 'Brasil', 'Manaus', 24, 'M', 'Pro', 'Desktop Gamer'),
    (18, 'Gvoy', 4, 'BR', 'Brasil', 'Salvador', 22, 'M', 'Gratuito', 'Desktop Gamer'),
    (19, 'Damage', 5, 'BR', 'Brasil', 'Fortaleza', 25, 'M', 'Gratuito', 'Notebook Gamer'),
    (20, 'Wizer', 8, 'KR', 'Coreia do Sul', 'Incheon', 26, 'M', 'VIP', 'Desktop Gamer'),
    (21, 'Crocodilo', 4, 'BR', 'Brasil', 'Recife', 22, 'M', 'Gratuito', 'Desktop Gamer'),
    (22, 'Smeb', 9, 'KR', 'Coreia do Sul', 'Seul', 29, 'M', 'Pro', 'Desktop Gamer'),
    (23, 'BeryL', 9, 'KR', 'Coreia do Sul', 'Daegu', 27, 'M', 'VIP', 'Desktop Gamer'),
    (24, 'Deft', 10, 'KR', 'Coreia do Sul', 'Seul', 28, 'M', 'VIP', 'Desktop Gamer'),
    (25, 'Keria', 10, 'KR', 'Coreia do Sul', 'Seul', 22, 'M', 'VIP', 'Desktop Gamer')
]

df_jogador = pd.DataFrame([
    {'id_jogador': p[0], 'nick': p[1], 'id_elo': p[2], 'regiao': p[3]}
    for p in jogadores_raw
])

df_demografia = pd.DataFrame([
    {
        'id_jogador': p[0], 'nick': p[1], 'regiao': p[3], 'pais': p[4],
        'cidade': p[5], 'idade': p[6], 'genero': p[7], 'tier_assinatura': p[8], 'plataforma': p[9]
    }
    for p in jogadores_raw
])

print(f"Total de Jogadores: {len(df_jogador)}")
df_demografia.head()

### 7. Partidas (`partida`) com Datas e Horários
Geramos 30 partidas com datas variando ao longo de dias da semana (Segunda a Domingo) e resultados (Vitória/Derrota). Essa temporalidade é essencial para responder aos requisitos de métricas do Spark (análise por dia da semana e filtragem temporal).

In [ ]:
base_date = datetime(2025, 8, 15)
partidas_data = []
resultados = ['Vitoria', 'Derrota']

for i in range(1, 31):
    dt = base_date + timedelta(days=(i * 2) % 36, hours=random.randint(10, 23), minutes=random.randint(0, 59))
    partidas_data.append({
        'id_partida': i,
        'data_partida': dt.strftime('%Y-%m-%d'),
        'hora_partida': dt.strftime('%H:%M:%S'),
        'duracao_minutos': random.randint(18, 45),
        'resultado': random.choice(resultados)
    })

df_partida = pd.DataFrame(partidas_data)
print(f"Total de Partidas geradas: {len(df_partida)}")
df_partida.head()

### 8. Fato Compras (`compra`) com Quantidade e Snapshot de Preço
A tabela fato central que registra as transações no contexto de cada partida, registrando jogador, item adquirido, data da compra, minuto do jogo, quantidade e total em ouro.

In [ ]:
item_ids_validos = df_tabela_item['item_id'].tolist()
item_price_map = dict(zip(df_tabela_item['item_id'], df_tabela_item['preco_unitario']))
jogadores_ids_validos = df_jogador['id_jogador'].tolist()

compras = []
id_compra = 1

for _, row_p in df_partida.iterrows():
    p_id = row_p['id_partida']
    p_data = row_p['data_partida']
    p_duracao = row_p['duracao_minutos']
    
    # 10 jogadores disputando a partida (5v5)
    participantes = random.sample(jogadores_ids_validos, 10)
    
    for jog in participantes:
        qtd_compras = random.randint(2, 5)
        for _ in range(qtd_compras):
            it_id = random.choice(item_ids_validos)
            p_unit = item_price_map[it_id]
            qtd = random.choices([1, 2], weights=[0.85, 0.15])[0]
            minuto = random.randint(1, p_duracao)
            
            compras.append({
                'id_compra': id_compra,
                'id_jogador': jog,
                'id_item': it_id,
                'id_partida': p_id,
                'data_compra': p_data,
                'minuto_compra': minuto,
                'quantidade': qtd,
                'preco_unitario': p_unit,
                'total_compra': p_unit * qtd
            })
            id_compra += 1

df_compra = pd.DataFrame(compras)
print(f"Total de compras geradas: {len(df_compra)}")
df_compra.head()

### 9. Exportação dos Datasets Estruturados
Exportamos todas as tabelas em formato CSV com cabeçalhos para o diretório `data/processed/`, prontas para carga no PostgreSQL, migração para MongoDB Atlas e ingestão no Apache Spark.

In [ ]:
caminho_processados = '../data/processed'
os.makedirs(caminho_processados, exist_ok=True)

df_elo.to_csv(f"{caminho_processados}/elo.csv", index=False)
df_categoria.to_csv(f"{caminho_processados}/categoria.csv", index=False)
df_tabela_item.to_csv(f"{caminho_processados}/item.csv", index=False)
df_item_categoria.to_csv(f"{caminho_processados}/item_categoria.csv", index=False)
df_jogador.to_csv(f"{caminho_processados}/jogador.csv", index=False)
df_demografia.to_csv(f"{caminho_processados}/jogadores_demografia.csv", index=False)
df_partida.to_csv(f"{caminho_processados}/partida.csv", index=False)
df_compra.to_csv(f"{caminho_processados}/compra.csv", index=False)

print("[OK] Sucesso! Todos os 8 arquivos CSV foram persistidos em data/processed/:")
for f in sorted(os.listdir(caminho_processados)):
    if f.endswith('.csv'):
        caminho = os.path.join(caminho_processados, f)
        print(f"   - {f:<25} ({os.path.getsize(caminho):>6} bytes)")